# course

> A course to your goal: its sources, its stages, where you are, what comes next and why. The module is `abhyasa.course`.

A goal such as "fine-tune my own diffusion model to make Indian comics" needs more than one walk. A **course** is all that
abhyasa has for one goal: its sources, its **stages** (one walk each), and your answers to "what do you know already?".

The course keeps only what abhyasa cannot get again from other records. "You are here", "next and why" and "what stands between
you and the goal" come from the walks, the concepts and the Leela results each time that you ask. So each fact has one place,
and no field gets old.

This module has no web page. The `course` block shows it.

- **The record**: one vault document of kind `course` on the learning shelf, as a walk is.
- **Join**: a goal walk makes a course, or joins the course of the same goal, as its first stage.
- **Where you start**: a few short questions from the model. Your answers set where the course starts and how many stages it has.
- **Stages**: the planner gives the stages. The next stage is built when the current one is 60% done, or when you open it.
- **The state**: `course_state` and `next_up`, with eight ordered rules, each with a one-line reason.

In [ ]:
#| default_exp course

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # nor the real ramabana config

In [ ]:
#| hide
from fastcore.test import *

Import the shelf, the learner model, the walk engine, the JSON call of the model and the guard for text that leaves the Mac.

In [ ]:
#| export
import json, re, threading, time
from fastcore.utils import *
from abhyasa.store import Learning, _src, fold
from abhyasa.model import learned, ready, recall, understanding
from abhyasa.settings import val
from abhyasa.llm import ask_json, for_task
from abhyasa.prompts import register, prompt
from abhyasa.ground import clip
from abhyasa.journey import walk_from_topic, walk_state, for_talk, _jlock, _new_id, keep_data
from abhyasa.walk import leela_found
from abhyasa.guide import guard

## The limits

`GROW_AT` is the share of a stage that you finish before abhyasa builds the next stage. `UNCLEAR` is the time in which two boards
or explainers on one idea make it unclear. A course keeps at most `ASKS_MOST` asks, the newest.

In [ ]:
#| export
GROW_AT = .6          # the share of the current stage that is done when the next stage is built
UNCLEAR = 2 * 86400.  # seconds: two boards or explainers on one idea in this time make the idea unclear
ASKS_MOST = 200       # the most asks that one course keeps, the newest
LEVEL_MOST = 5        # the most questions of "where do you start"
ANSWERS = dict(yes='Yes', some='A little', no='No')   # an answer to a level question -> its word on the page

## The record

A course is one document of kind `course` on the learning shelf. Its fields:

| field | meaning |
|---|---|
| `id`, `goal`, `title`, `subject`, `makes` | the goal is your words, as you typed them |
| `sources` | `{id, kind, root, count, status, why}` for each chip: `ready`, `filing` or `failed` |
| `stages` | `{key, name, makes, ideas, foundation, skip_if, short, walk, status}` in order: `planned`, `building`, `ready` or `skipped` |
| `level` | `{at, qs: [{key, q, idea, answer, by}]}`: the questions of "where do you start", and your answers |
| `here` | `{walk, step, slug, at}`: the last idea that you opened |
| `asks` | `{at, where, said, slug, kind, ref, outcome}`: boards, explainers and demos that you asked for |
| `proposal` | ideas that a new source adds, not applied yet |
| `v` | the count of writes |

`new_course` makes the fields. `put_course` saves a course and replaces the one with the same id.

In [ ]:
#| export
def new_course(goal:str, title:str='', subject:str='', sources=()) -> dict:
    "The fields of a new course to `goal`, with no stage and no data folder."
    goal = ' '.join(str(goal or '').split())
    title = title or clip(goal, 60)
    return dict(id=_new_id(title), goal=goal, title=title, subject=subject, makes='', sources=list(sources), data=[], stages=[], level=None,
                here=None, asks=[], proposal=None, done_at=None, v=0, created=time.time())

@patch
def put_course(self:Learning, c:dict) -> AttrDict:
    "Save course `c`, and replace an earlier one with the same id."
    c = {k: v for k, v in dict(c).items() if k != 'doc_id'}
    c.update(updated=time.time(), v=(c.get('v') or 0) + 1)
    text = f"Course: {c['title']} ({c['id']})\n\nGoal: {c.get('goal', '')}\n\nStages: {', '.join(s['name'] for s in c.get('stages') or ())}"
    r = self.v.add(text, title=f"Course: {c['title']}", source=_src('course', c['id']), kind='course', meta=c)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **c)
    return self.course(c['id'])

def _cmeta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def course(self:Learning, cid:str) -> AttrDict|None:
    "One course by id."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('course', cid)]))
    return _cmeta(r) if r else None

@patch
def courses(self:Learning) -> L:
    "All courses, the most recently changed first."
    return L(sorted(map(_cmeta, self._docs('course')), key=lambda c: -c.get('updated', 0)))

@patch
def drop_course(self:Learning, cid:str):
    "Remove a course. Its walks and concepts stay."
    if (c := self.course(cid)): self.v.forget(c.doc_id)

`merge_course` and `change_course` change a course under its lock, so two requests do not lose a change.

In [ ]:
#| export
def _clock(cid): return _jlock(f'course:{cid}')   # the lock of one course, in the same table as the locks of the walks

@patch
def merge_course(self:Learning, cid:str, **kv) -> AttrDict:
    "Merge the fields `kv` into course `cid` and save it, under the lock of the course."
    with _clock(cid):
        c = self.course(cid)
        if c is None: raise KeyError(f'no course {cid!r}')
        return self.put_course({**c, **kv})

@patch
def change_course(self:Learning, cid:str, f) -> AttrDict:
    "Call `f(course)` under the lock of course `cid`, and save the course that it changed."
    with _clock(cid):
        c = self.course(cid)
        if c is None: raise KeyError(f'no course {cid!r}')
        f(c)
        return self.put_course(c)

A course saves, reads, and changes under its lock. Each write adds one to `v`.

In [ ]:
from abhyasa.store import temp_learning

In [ ]:
lr = temp_learning()
c0 = lr.put_course(new_course('Make  a  comic', title='Comics'))
test_eq((c0.goal, c0.v, lr.courses().attrgot('id')), ('Make a comic', 1, [c0.id]))
test_eq(lr.merge_course(c0.id, makes='One panel').v, 2)
test_eq(lr.change_course(c0.id, lambda c: c.stages.append(dict(key='s1', name='One'))).stages[0]['name'], 'One')
lr.drop_course(c0.id); test_eq(lr.courses(), [])

## A goal walk joins a course

A walk that you plan from a goal (`from_goal` in the walk block) is a **goal walk**: a walk with a goal, no source document, no
parent, and not for a talk. `join_course` gives it a course: the course of the same goal, else a new one. The walk is a stage of
the course, and the walk knows its course (`course`, `stage`).

`chip_sources` makes the source rows from the chips of the walk. A chip that did not file is `failed`, with the line that the
walk kept.

In [ ]:
#| export
KINDS = dict(gh='a GitHub repo', folder='a folder', repo='a folder', link='a link', vault='your vault', doc='a document')

def is_goal_walk(j) -> bool:
    "Whether `j` is a walk planned from a goal: a walk with a goal, no source document and no parent, not for a talk."
    return bool(j) and j.get('mode') == 'walk' and bool(str(j.get('goal') or '').strip()) and not j.get('source') and not j.get('parent') and not for_talk(j)

def chip_sources(j) -> list:
    "A source row `{id, kind, root, count, status, why}` for each chip of walk `j`. A chip with no count did not file."
    counts, notes, out = j.get('chip_counts') or {}, j.get('notes') or [], []
    for i in j.get('chips') or ():
        kind, _, v = str(i).partition(':')
        root = v.rstrip('/').rsplit('/', 1)[-1] if kind in ('folder', 'repo') else v
        ok = i in counts
        why = '' if ok else first(n for n in notes if v in n or root in n) or 'abhyasa could not file it.'
        out.append(dict(id=i, kind=KINDS.get(kind, kind), root=root, count=counts.get(i, 0), status='ready' if ok else 'failed', why=why, added_at=j.get('created')))
    return out

def _same(a, b) -> bool: return fold(' '.join(str(a or '').split())) == fold(' '.join(str(b or '').split()))

def _next_key(stages) -> str:
    "A stage key that `stages` does not have: `s1`, `s2` and so on."
    ns = [int(m[1]) for s in stages if (m := re.fullmatch(r's(\d+)', str(s.get('key'))))]
    return f's{max(ns, default=0) + 1}'

In [ ]:
#| export
_JOIN = threading.Lock()

def join_course(lr:Learning, j) -> AttrDict|None:
    "The course of goal walk `j` (a walk or its id): its own, else the course of the same goal, else a new one. The walk becomes a stage. None for a walk that is not a goal walk."
    j = lr.journey(j if isinstance(j, str) else j['id'])
    if not is_goal_walk(j): return None
    if j.get('course') and (c := lr.course(j.course)) and any(s.get('walk') == j.id for s in c.stages): return c
    with _JOIN:   # two requests for one goal make one course
        c = first(x for x in lr.courses() if _same(x.goal, j.goal)) or lr.put_course(new_course(j.goal, j.title, j.get('subject') or ''))
        key = first(s['key'] for s in c.stages if s.get('walk') == j.id)
        if key is None:
            key = _next_key(c.stages)
            makes = first(s.get('makes') for s in reversed(j.steps) if s.get('makes')) or ''
            def add(c):
                have = {s['id'] for s in c.sources}
                c.sources += [s for s in chip_sources(j) if s['id'] not in have]
                c.stages.append(dict(key=key, name=j.title, makes=makes, ideas=[], foundation=False, skip_if='', short=False, walk=j.id, status='ready'))
            c = lr.change_course(c.id, add)
        lr.merge_journey(j.id, course=c.id, stage=key, **({'data': c.data} if c.get('data') and not j.get('data') else {}))   # a walk that joins gets your data
        return lr.course(c.id)

def course_of(lr:Learning, jid:str) -> AttrDict|None:
    "The course that walk `jid` is a stage of, or None."
    j = lr.journey(jid)
    c = lr.course(j.course) if j and j.get('course') else None
    return c if c and any(s.get('walk') == jid for s in c.stages) else None

def ensure_courses(lr:Learning) -> list:
    "Give a course to each goal walk from the Start box (a walk with `chips`) that has none. Return the ids of the courses that changed."
    return [c.id for j in lr.journeys() if is_goal_walk(j) and 'chips' in j and not j.get('course') and (c := join_course(lr, j))]

A small goal walk, as the Start box makes it: two chips, and the clone of one failed.

In [ ]:
lr = temp_learning()
for n, pre in (('Noise', []), ('Denoise', ['noise']), ('Sample', ['denoise'])): lr.put(n, f'{n} a picture.', 'ml', prereqs=pre)
def goal_walk(lr, jid='gw', goal='Fine-tune diffusion for comics', slugs=('noise', 'denoise', 'sample'), **kw):
    return lr.put_journey(dict(id=jid, title='Diffusion for comics', goal=goal, mode='walk', source=None, started=None, created=1.,
                               steps=[dict(slug=s, status='todo', check=dict(way='words', kind='short', depth=2)) for s in slugs],
                               chips=['folder:/x/course22p2', 'gh:huggingface/diffusers'], chip_counts={'folder:/x/course22p2': 44},
                               notes=['abhyasa could not clone huggingface/diffusers.'], **kw))
gw = goal_walk(lr)

The walk joins a new course as stage `s1`. The failed clone is a `failed` source with its line. A second call gives the same course.

In [ ]:
c = join_course(lr, gw)
test_eq([(s['key'], s['walk'], s['status']) for s in c.stages], [('s1', 'gw', 'ready')])
test_eq([(s['root'], s['status']) for s in c.sources], [('course22p2', 'ready'), ('huggingface/diffusers', 'failed')])
test_eq(c.sources[1]['why'], 'abhyasa could not clone huggingface/diffusers.')
test_eq((lr.journey('gw').course, lr.journey('gw').stage), (c.id, 's1'))
test_eq(join_course(lr, 'gw').id, c.id); test_eq(len(lr.courses()), 1); test_eq(course_of(lr, 'gw').id, c.id)

A second goal walk with the same goal (case and spaces do not count) joins the same course as stage `s2`. A side walk joins nothing.

In [ ]:
goal_walk(lr, 'gw2', goal='fine-tune  Diffusion for comics')
test_eq(join_course(lr, 'gw2').stages[-1]['key'], 's2'); test_eq(len(lr.courses()), 1)
lr.put_journey(dict(id='side', title='S', goal='x', mode='walk', source=None, parent=dict(walk='gw', step=0), steps=[]))
test_is(join_course(lr, 'side'), None)

`ensure_courses` gives a course to an older goal walk that has none, one time.

In [ ]:
l2 = temp_learning(); l2.put('Noise', 'n', 'ml'); goal_walk(l2, slugs=('noise',))
test_eq(len(ensure_courses(l2)), 1); test_eq(ensure_courses(l2), [])

## Where do you start

Before abhyasa plans the stages, it asks what you know already: a few short questions, such as "Do you know how a neural network
learns?". You answer each with yes, a little, or no, by a tap or by voice. The model writes the questions from the goal. An idea
that you learned already (a concept at the learned depth) gets the answer yes from the shelf, and abhyasa does not ask it.

The prompt goes through the guard of ramabana (`guard`), as the other calls do, so no secret or personal data leaves the Mac.

In [ ]:
#| export
LEVEL_SP = register('course.level', 'Where you start', 'How the tutor asks what you know before it plans the stages of a course.', """You find out what one learner knows before you plan a course to their goal.
- Ask 2 to 5 short questions. The learner answers each one with yes, a little, or no.
- Each question asks about one idea that the goal needs and that some learners know already. Go from the most basic idea to the most advanced.
- Write each question as "Do you know ...?", to the learner as "you", in ASD-STE100 Simplified Technical English, in 12 words or fewer.
- `idea` is the name of the idea, in 2 to 6 words.
- Do not ask about the concepts that the learner knows (the list).""")

def level_prompt(goal:str, who:str='', known=()) -> str:
    "The prompt that asks for the questions of \"where do you start\" for `goal`. `known` are the names of the concepts that you learned."
    kn = '\n'.join(f'- {n}' for n in known)
    return (f"Goal: {goal}\n" + (f"About the learner:\n{who}\n" if who else '') + (f"Concepts the learner knows:\n{kn}\n" if kn else '')
            + '\nReturn JSON: {"questions": [{"q": str, "idea": str}]}')

def _guarded(ask): return lambda p, sp='': ask(guard(p), guard(sp))   # the text that leaves the Mac passes the guard of ramabana

def _known(lr:Learning) -> dict:
    "Folded name -> concept, for each concept that you learned."
    return {fold(c.name): c for c in lr.concepts() if learned(c)}

def level_questions(lr:Learning, ask, goal:str, who:str='') -> list:
    "The questions of \"where do you start\": `{key, q, idea, answer, by}`. An idea that you learned has the answer yes, by the shelf."
    kn = _known(lr)
    r = ask_json(_guarded(for_task(ask, 'plan')), level_prompt(goal, who, [c.name for c in kn.values()][:60]), prompt('course.level'))
    out = []
    for x in (r.get('questions') if isinstance(r, dict) else r) or ():
        if not (isinstance(x, dict) and isinstance(x.get('q'), str) and x['q'].strip()): continue
        idea = clip(str(x.get('idea') or '').strip(), 80)
        shelf = bool(idea) and fold(idea) in kn
        out.append(dict(key=f'q{len(out) + 1}', q=clip(' '.join(x['q'].split()), 160), idea=idea, answer='yes' if shelf else None, by='shelf' if shelf else ''))
    if not out: raise ValueError('the model gave no questions')
    return out[:LEVEL_MOST]

`read_answer` reads an answer that you said or typed: "yes, I know it", "a little" or "no, never". It gives None for other words.

In [ ]:
#| export
SAID = dict(some=('a little', 'a bit', 'little', 'some', 'partly', 'sort of', 'kind of', 'not much', 'not well'),
            no=('no', 'nope', 'not', 'never', 'dont', 'none', 'nothing'),
            yes=('yes', 'yeah', 'yep', 'sure', 'i do', 'i know', 'of course', 'well'))

def read_answer(text) -> str|None:
    "`yes`, `some` or `no` from an answer in words, or the answer itself; None when the words say none of them."
    t = str(text or '').strip().lower()
    if t in ANSWERS: return t
    t = ' ' + ' '.join(re.findall(r"[a-z]+", t.replace("'", ''))) + ' '
    return first(k for k in ('some', 'no', 'yes') if any(f' {w} ' in t for w in SAID[k]))

In [ ]:
test_eq([read_answer(x) for x in ('Yes, I know it', 'a little bit', "No, I don't", 'not well', 'some', 'banana', 'yes')], ['yes', 'some', 'no', 'some', 'some', None, 'yes'])

`ask_level` keeps the questions on the course, with no answers. `answer_level` keeps one answer. `level_done` is true when each question has an answer.

In [ ]:
#| export
def ask_level(lr:Learning, ask, cid:str, who:str='') -> AttrDict:
    "Write the questions of \"where do you start\" for course `cid` and keep them on it. Older answers go."
    c = lr.course(cid)
    if c is None: raise KeyError(f'no course {cid!r}')
    return lr.merge_course(cid, level=dict(at=time.time(), qs=level_questions(lr, ask, c.goal, who)))

def answer_level(lr:Learning, cid:str, key:str, answer) -> AttrDict:
    "Keep the answer to question `key` (yes, some, no, or words that `read_answer` reads). A bad answer raises `ValueError`."
    if (a := read_answer(answer)) is None: raise ValueError('Answer yes, a little, or no.')
    def f(c):
        q = first(q for q in (c.get('level') or {}).get('qs') or () if q['key'] == key)
        if q is None: raise KeyError(f'no question {key!r}')
        q.update(answer=a, by='you', at=time.time())
    return lr.change_course(cid, f)

def level_done(c) -> bool:
    "Whether course `c` has questions and an answer to each one."
    qs = (c.get('level') or {}).get('qs') or ()
    return bool(qs) and all(q.get('answer') for q in qs)

A fake model writes three questions. You learned Noise already, so its question has the answer yes from the shelf.

In [ ]:
from abhyasa.llm import Fake

In [ ]:
LEVEL = dict(questions=[dict(q='Do you know how a neural network learns?', idea='How a neural network learns'),
                        dict(q='Do you know what noise is?', idea='Noise'), dict(q='Have you used diffusers?', idea='The diffusers library')])
lr.update('noise', floor=4, seen=1)
ask = Fake({'You find out what one learner knows': LEVEL})
c = ask_level(lr, ask, c.id)
test_eq([(q['key'], q['answer'], q['by']) for q in c.level['qs']], [('q1', None, ''), ('q2', 'yes', 'shelf'), ('q3', None, '')])
test_eq(level_done(c), False)
c = answer_level(lr, c.id, 'q1', 'no, never'); c = answer_level(lr, c.id, 'q3', 'a little')
test_eq((c.level['qs'][0]['answer'], level_done(c)), ('no', True))
test_fail(lambda: answer_level(lr, c.id, 'q1', 'banana'), contains='yes, a little, or no')

## Stages

`plan_stages` asks the planner for the stages of the course. It gives the planner your answers and the stages that have a walk.
The planner marks a stage that teaches the basics as `foundation`, with `skip_if`: the question that makes it not needed.

Then the rules decide, not the model:

- A foundation stage whose question you answered **yes** is `skipped`. It shows, but it counts nothing.
- A foundation stage whose question you answered **a little** is `short`: a refresher of 3 or 4 ideas.
- A foundation stage comes before the stages that have a walk. The other new stages come after them.
- A stage that has a walk stays as it is. The stages with no walk are planned again each time.

So the answers set where the course starts and how many stages it has.

In [ ]:
#| export
STAGES_SP = register('course.stages', 'Course planner', 'How the tutor plans the stages of a course to your goal.', """You plan a course to one learner's goal, as stages.
- A stage makes one thing that the goal needs, such as "a pipeline that draws 4 images". Give 2 to 6 stages, in the order to learn them.
- Each stage has 3 to 12 ideas. An idea is one thing that the learner can explain.
- The learner told you what they know. Do not give a stage that teaches only what they know.
- A stage that teaches the basics that the goal needs has `foundation`: true, and `skip_if`: the key of the question (such as "q1") that, with the answer yes, makes the stage not needed. Else `skip_if` is "".
- Do not give again the stages that the course has now.
- `makes` says what the learner has at the end of the stage, in a few words. Write in ASD-STE100 Simplified Technical English.""")

def stages_prompt(c, who:str='') -> str:
    "The prompt that plans the stages of course `c`, with the answers of the learner and the stages that have a walk."
    lv = '\n'.join(f"- {q['key']}: {q['q']} Answer: {ANSWERS.get(q.get('answer'), 'no answer')}" for q in (c.get('level') or {}).get('qs') or ())
    have = '\n'.join(f"- {s['name']}" + (f" (makes {s['makes']})" if s.get('makes') else '') for s in c.get('stages') or () if s.get('walk'))
    return (f"Goal: {c['goal']}\n" + (f"About the learner:\n{who}\n" if who else '') + (f"What the learner told you:\n{lv}\n" if lv else '')
            + (f"The stages that the course has now:\n{have}\n" if have else '')
            + '\nReturn JSON: {"title": str (3 to 6 words), "makes": str, "stages": [{"name": str, "makes": str, "ideas": [str], "foundation": bool, "skip_if": str}]}')

In [ ]:
#| export
def _stage(x, key, q) -> dict|None:
    "The stage from the planned stage `x`, with the question `q` of its `skip_if`; None when `x` is not a stage."
    if not (isinstance(x, dict) and isinstance(x.get('name'), str) and x['name'].strip()): return None
    base, ans = x.get('foundation') is True, (q or {}).get('answer')
    ideas = [clip(' '.join(i.split()), 80) for i in x.get('ideas') or () if isinstance(i, str) and i.strip()][:val('walks.steps')]
    return dict(key=key, name=clip(' '.join(x['name'].split()), 80), makes=clip(str(x.get('makes') or '').strip(), 120), ideas=ideas,
                foundation=base, skip_if=(q or {}).get('key', ''), short=base and ans == 'some', walk=None,
                status='skipped' if base and ans == 'yes' else 'planned', why=f"You know it: {q['idea'] or q['q']}" if base and ans == 'yes' else '')

def plan_stages(lr:Learning, ask, cid:str, who:str='') -> AttrDict:
    "Plan the stages of course `cid` with the planner and your answers. The stages with a walk stay; the others are planned again."
    c = lr.course(cid)
    if c is None: raise KeyError(f'no course {cid!r}')
    r = ask_json(_guarded(for_task(ask, 'plan')), stages_prompt(c, who), prompt('course.stages'))
    qs = (c.get('level') or {}).get('qs') or ()
    by_key, by_idea = {q['key']: q for q in qs}, {fold(q.get('idea')): q for q in qs if q.get('idea')}
    def f(c):
        built = [s for s in c.stages if s.get('walk')]
        have, front, back, ks = {fold(s['name']) for s in built}, [], [], list(built)
        for x in (r.get('stages') if isinstance(r, dict) else None) or ():
            sk = str(x.get('skip_if') or '') if isinstance(x, dict) else ''
            if not (st := _stage(x, _next_key(ks), by_key.get(sk) or by_idea.get(fold(sk)))) or fold(st['name']) in have: continue
            ks.append(st); have.add(fold(st['name']))
            (front if st['foundation'] else back).append(st)
        c.stages = front + built + back
        c.update(makes=c.get('makes') or clip(str(r.get('makes') or ''), 120), planned_at=time.time())
    return lr.change_course(cid, f)

A fake planner gives three stages for the diffusion goal. The first teaches how a neural network learns, and is skipped when
you answer yes to `q1`.

In [ ]:
STAGES = dict(title='Diffusion for comics', makes='a LoRA in your comic style', stages=[
    dict(name='How a neural network learns', makes='a small net that learns', ideas=['Gradient descent', 'Loss'], foundation=True, skip_if='q1'),
    dict(name='Diffusion for comics', makes='x', ideas=[]),   # the stage that has a walk: not again
    dict(name='Your comic data', makes='captions.jsonl', ideas=['Captions', 'Image size'])])
ask = Fake({'You find out what one learner knows': LEVEL, 'You plan a course to one learner': STAGES})
def course_with(answer):
    l = temp_learning(); l.put('Noise', 'n', 'ml'); goal_walk(l, slugs=('noise',))
    c = join_course(l, 'gw'); ask_level(l, ask, c.id)
    for k in ('q1', 'q2', 'q3'): answer_level(l, c.id, k, answer)
    return l, plan_stages(l, ask, c.id)

A learner who does **not** know how a neural network learns gets the foundation stage first, before the goal walk.

In [ ]:
l, c = course_with('no')
test_eq([(s['name'], s['status']) for s in c.stages], [('How a neural network learns', 'planned'), ('Diffusion for comics', 'ready'), ('Your comic data', 'planned')])
test_eq(c.makes, 'a LoRA in your comic style')

A learner who knows it skips the foundation stage. A learner who knows it a little gets a short one.

In [ ]:
l, c = course_with('yes')
test_eq([(s['name'], s['status']) for s in c.stages][0], ('How a neural network learns', 'skipped'))
test_eq(c.stages[0]['why'], 'You know it: How a neural network learns')
test_eq(course_with('some')[1].stages[0]['short'], True)

## Build a stage

`build_stage` makes the walk of one planned stage. It also makes the walk of a skipped stage when you ask for it, and the stage stays skipped if this fails. The planner gets a brief: the goal, the stage, its ideas, and the ideas of the
earlier stages, which it leaves out. A step on a concept of an earlier stage goes, so a stage never asks an idea again.
The walk knows its course and stage, and it reads the sources of the course.

`due_stage` says which stage to build now: the current stage when it has no walk, or the next stage when the current one is
`GROW_AT` (60%) done.

In [ ]:
#| export
def _slugs(lr:Learning, c, before:str=None) -> list:
    "The concepts of the steps of the stages of `c` that have a walk, up to stage `before`."
    out = []
    for s in c.stages:
        if s['key'] == before: break
        if s.get('walk') and (j := lr.journey(s['walk'])): out += [x['slug'] for x in j.steps if x['slug'] not in out]
    return out

def stage_brief(lr:Learning, c, st) -> str:
    "The goal for the planner of stage `st` of course `c`."
    names = [(lr.concept(s) or {}).get('name', s) for s in _slugs(lr, c)]
    return (f"{c.goal}\nThis is one stage of a course to this goal. The stage: {st['name']}." + (f" It makes: {st['makes']}." if st.get('makes') else '')
            + (f" Its ideas: {', '.join(st['ideas'])}." if st.get('ideas') else '')
            + (' Keep it short: a refresher of 3 or 4 ideas.' if st.get('short') else '')
            + (f"\nEarlier stages teach these ideas, so leave them out: {', '.join(names[:60])}." if names else ''))

def _set_stage(lr, cid, key, **kv):
    def f(c):
        if (s := first(s for s in c.stages if s['key'] == key)) is not None: s.update(kv)
    return lr.change_course(cid, f)

def build_stage(lr:Learning, ask, cid:str, key:str, sources=None, who:str='') -> AttrDict:
    """Make the walk of stage `key` of course `cid`, or give the walk that it has. On a failure the stage is `planned` again, and the error goes up.
    A skipped stage stays `skipped` until its walk is made: a failure does not change it."""
    c = lr.course(cid)
    if c is None or (st := first(s for s in c.stages if s['key'] == key)) is None: raise KeyError(f'no stage {key!r} in course {cid!r}')
    if st.get('walk') and (j := lr.journey(st['walk'])): return j
    skip = st.get('status') == 'skipped'
    if not skip: _set_stage(lr, cid, key, status='building', why='')
    try:
        chips = [s['id'] for s in c.sources if s.get('status') == 'ready']
        j = walk_from_topic(lr, ask, stage_brief(lr, c, st), who, sources, chips=chips, data=c.get('data') or ())
        seen = set(_slugs(lr, c))
        steps = [s for s in j.steps if s['slug'] not in seen]
        if not steps:
            lr.drop_journey(j.id)
            raise ValueError('The planner gave no new ideas for this stage.')
        j = lr.merge_journey(j.id, steps=steps, title=st['name'], goal=c.goal, course=cid, stage=key, chips=chips)
    except Exception as e:
        if not skip: _set_stage(lr, cid, key, status='planned', why=str(e) if isinstance(e, ValueError) else 'abhyasa could not plan this stage. Try again.')
        raise
    _set_stage(lr, cid, key, walk=j.id, status='ready')
    return j

A fake planner gives the walk of the data stage. It names Noise again, which stage 1 has, so that step goes.

In [ ]:
PLAN = dict(title='Data', subject='ml', steps=[dict(name='Noise', statement='n', why='w'), dict(name='Captions', statement='Words for each picture.', why='w'),
                                               dict(name='Image size', statement='512 pixels.', why='w')])
ask.rules['You plan a learning path'] = PLAN
l, c = course_with('yes')
data = first(s for s in c.stages if s['name'] == 'Your comic data')
sj = build_stage(l, ask, c.id, data['key'])
test_eq([s['slug'] for s in sj.steps], ['captions', 'image-size'])
test_eq((sj.course, sj.stage, sj.goal, sj.title), (c.id, data['key'], c.goal, 'Your comic data'))
test_eq(first(s for s in l.course(c.id).stages if s['key'] == data['key'])['status'], 'ready')
test_eq(build_stage(l, ask, c.id, data['key']).id, sj.id)   # a second call gives the same walk
test_is('Earlier stages teach these ideas, so leave them out: Noise.' in stage_brief(l, c, data), True)

### Your data on a course

`add_data` keeps a folder on the course as your data, with a note. abhyasa does not file it and does not learn from it: the
exercises run on it. The walks of the stages keep the same list, so the notebook of each step in Leela gets it. A walk that a
stage makes later gets it from `build_stage`. `drop_data` removes one folder from the course and its walks.

In [ ]:
#| export
def _set_data(lr:Learning, cid:str, f) -> AttrDict:
    "Change the data folders of course `cid` with `f` (the old list to the new list) under its lock. The walks of its stages get the new list."
    c = lr.change_course(cid, lambda c: c.update(data=f(c.get('data') or [])))
    for s in c.stages:
        if s.get('walk') and lr.journey(s['walk']): lr.merge_journey(s['walk'], data=c.data)
    return c

def add_data(lr:Learning, cid:str, path, note:str='') -> AttrDict:
    "Keep the folder `path` with `note` on course `cid` and its walks as your data. A `ValueError` says in one line why abhyasa cannot."
    return _set_data(lr, cid, lambda ds: keep_data(ds, path, note))

def drop_data(lr:Learning, cid:str, path:str) -> AttrDict:
    "Remove the data folder `path` from course `cid` and its walks. The folder stays on the disk."
    return _set_data(lr, cid, lambda ds: [d for d in ds if d['path'] != path])

A folder added as data is on the course and on its walk. The walk that a stage makes after that keeps it, and the planner sees it in the data list, with a listing of the folder. A path that is not a folder is refused. `drop_data` removes it from the course and from each walk.

In [ ]:
home, old = Path(tempfile.mkdtemp()).resolve(), os.environ['HOME']
os.environ['HOME'] = str(home)
try:
    (home/'episodes').mkdir(); (home/'episodes'/'episodes.json').write_text('{}')
    l2, c2 = course_with('yes')
    c2 = add_data(l2, c2.id, str(home/'episodes'), 'The episodes that my scenarios run on.')
    with ExceptionExpected(ValueError, 'no folder'): add_data(l2, c2.id, '~/nothing')
    j2 = build_stage(l2, ask, c2.id, first(s for s in c2.stages if s['name'] == 'Your comic data')['key'])
finally: os.environ['HOME'] = old
mine = [dict(path='~/episodes', note='The episodes that my scenarios run on.')]
test_eq((c2.data, l2.journey('gw').data, j2.data), (mine, mine, mine))
assert '- EPISODES: The episodes that my scenarios run on. It holds: episodes.json.' in first(p for p, sp in reversed(ask.calls) if 'You plan a learning path' in p + sp)
c2 = drop_data(l2, c2.id, '~/episodes')
test_eq((c2.data, l2.journey('gw').data, l2.journey(j2.id).data), ([], [], []))

A skipped stage stays skipped when its build fails. Here the planner gives only Noise again, so the stage gets no new ideas.

In [ ]:
sl, sc = course_with('yes')
k0 = sc.stages[0]['key']
ask.rules['You plan a learning path'] = dict(title='Nets', subject='ml', steps=[dict(name='Noise', statement='n', why='w')])
test_fail(lambda: build_stage(sl, ask, sc.id, k0), contains='no new ideas')
test_eq({k: sl.course(sc.id).stages[0][k] for k in ('status', 'why')}, dict(status='skipped', why='You know it: How a neural network learns'))

When the build of a skipped stage works, the stage gets its walk and is ready.

In [ ]:
ask.rules['You plan a learning path'] = dict(title='Nets', subject='ml', steps=[dict(name='Loss', statement='How wrong the net is.', why='w')])
test_eq((build_stage(sl, ask, sc.id, k0).stage, sl.course(sc.id).stages[0]['status']), (k0, 'ready'))
ask.rules['You plan a learning path'] = PLAN

## The state of a course

`stage_rows` gives each stage with its walk, its counts and one state word: `done`, `ready` (it has a walk, and ideas to learn),
`planned`, `building`, `skipped`, or `known`. A planned stage whose ideas you learned in the meantime is `known`: the plan changes
when what you show in checks changes. The **current stage** is the first stage that is `ready`, `planned` or `building`.

In [ ]:
#| export
LEFT = ('ready', 'planned', 'building')   # the states of a stage that is not finished

def stage_rows(lr:Learning, c, by:dict=None) -> list:
    "Each stage of course `c` with `n`, its walk `j`, its `walk_state` `ws`, `done`, `total` and `state`."
    by = by if by is not None else {x.slug: x for x in lr.concepts()}
    kn, out = {fold(x.name) for x in by.values() if learned(x)}, []
    for k, s in enumerate(c.get('stages') or ()):
        j = lr.journey(s['walk']) if s.get('walk') else None
        ws = walk_state(j, by) if j else None
        done, total = (ws.done, ws.total) if ws else (0, 0 if s.get('status') == 'skipped' else len(s.get('ideas') or ()) or val('walks.steps'))
        if s.get('status') == 'skipped': state = 'skipped'
        elif ws: state = 'done' if ws.current is None and j.steps else 'ready'
        elif s.get('ideas') and all(fold(i) in kn for i in s['ideas']): state, done = 'known', total
        else: state = s.get('status') if s.get('status') in LEFT else 'planned'
        out.append(AttrDict(s, n=k + 1, j=j, ws=ws, done=done, total=total, state=state))
    return out

def due_stage(lr:Learning, c, rows=None) -> str|None:
    "The key of the stage to build now: the current stage when it has no walk, else the next planned stage when the current one is `GROW_AT` done."
    rows = rows if rows is not None else stage_rows(lr, c)
    k = next((k for k, r in enumerate(rows) if r.state in LEFT), None)
    if k is None or rows[k].state == 'building': return None
    if rows[k].state == 'planned': return rows[k]['key']
    if rows[k].total and rows[k].done / rows[k].total < GROW_AT: return None
    return first(r['key'] for r in rows[k + 1:] if r.state == 'planned')

The data stage has a walk now. Stage 1 is skipped. Stage 2 (the goal walk) is current and less than 60% done, so `due_stage` names nothing to build.

In [ ]:
rows = stage_rows(l, l.course(c.id))
test_eq([(r.n, r.state) for r in rows], [(1, 'skipped'), (2, 'ready'), (3, 'ready')])
test_eq(due_stage(l, l.course(c.id)), None)

In the course of a learner who said no, the foundation stage is current and has no walk, so it is due at once. A planned stage whose ideas you learned is `known`.

In [ ]:
l, c = course_with('no')
test_eq(due_stage(l, c), 's2')
for n in ('Gradient descent', 'Loss'): l.put(n, n, 'ml'); l.update(l.concepts().filter(lambda x: x.name == n)[0].slug, floor=4, seen=1)
test_eq(stage_rows(l, c)[0].state, 'known'); test_eq(due_stage(l, c), None)

## What to learn next, and why

`next_up` gives one thing to do and one reason. The first rule that gives a thing wins:

| # | rule | reason |
|---|---|---|
| 1 | Leela filed a failed check on an idea of the current stage | "Your notebook did not pass. Look at the error first." |
| 2 | A review is due for an idea that the current stage needs, and its recall is under `review.fading` | "Review Softmax first. The next idea needs it, and you start to forget it." |
| 3 | You asked about an idea of the stage two or more times in two days (boards, explainers), with no right check since | "You asked about alpha bar two times. Check it now." |
| 4 | The next idea is ready: you know the ideas that it needs | "Next: Add noise at step t. You know the ideas that it needs." |
| 5 | The next idea is not ready | "Next: Derivatives first. Add noise at step t needs it." |
| 6 | The current stage has no walk, and you are weak on an idea that it needs | "Practice Guidance. Stage 4 builds on it." |
| 7 | The current stage has no walk yet | "Stage 3 is next: Your comic data. It makes captions.jsonl." |
| 8 | All stages are done | "You reached your goal." |

Rules for all: only one review before a new idea, so the course moves (a review since the last idea that you finished turns rule 2
off). A demo or an explainer never counts as learned. When you opened an idea that is not the first one to learn, it wins once, and
the reason says "You opened this idea last, so you go on from it."

`go` says where the button goes: `{kind: 'step', walk, step}`, `{kind: 'review', slug}`, `{kind: 'stage', key}` or `{kind: 'done'}`.
The block makes the link.

In [ ]:
#| export
def _n(n, w) -> str: return f"{n} {w if n == 1 else w + 's'}"
def _times(n) -> str: return {2: 'two', 3: 'three', 4: 'four', 5: 'five'}.get(n, str(n))
def _name(by, slug) -> str: return (by.get(slug) or {}).get('name') or slug
def _left(r) -> list: return [i for i, s in enumerate(r.ws.steps) if s.dot in ('now', 'todo')] if r.ws else []
def _step(r, i): return dict(kind='step', walk=r.j.id, step=i)
def _done_at(j) -> float: return max([(s.get('done') or {}).get('at') or 0 for s in j.steps] + [j.get('started') or 0])

def _here(c, r) -> int|None:
    "The step of stage row `r` that you opened last (`here`), when it is still to learn; else None."
    h = c.get('here') or {}
    if not r.ws or h.get('walk') != r.j.id: return None
    i = next((k for k, s in enumerate(r.j.steps) if s['slug'] == h.get('slug')), h.get('step'))
    return i if isinstance(i, int) and i in _left(r) else None

In [ ]:
#| export
def _leela_failed(lr, r):
    "Rule 1: the first step of stage row `r` with a failed check from Leela and no outcome."
    for i, b in sorted(leela_found(lr, r.j).items()):
        if b.check and b.check.get('correct') is False and not r.j.steps[i].get('done'): return i
    return None

def _review(lr, r, by, now):
    "Rule 2: the concept with the lowest recall that the stage needs, with a review due and recall under `review.fading`; None after one review since the last idea."
    left = [r.j.steps[i]['slug'] for i in _left(r)]
    need = {p for s in left for p in (by.get(s) or {}).get('prereqs') or ()} | {s['slug'] for s in r.j.steps if s.get('done')}
    due = [by[x] for x in need if x in by and by[x].get('seen') and (by[x].get('due') or now + 1) <= now and recall(by[x], now) < val('review.fading')]
    if not due: return None
    since = _done_at(r.j)
    if any(lr.attempts(x.slug, limit=1, since=since) for x in due if since): return None   # one review before a new idea
    return min(due, key=lambda x: recall(x, now))

def _unclear(lr, c, r, now):
    "Rule 3: `(step, count)` for an idea of the stage with two or more boards or explainers in `UNCLEAR`, and no right check after the last one."
    slugs = [s['slug'] for s in r.j.steps]
    asks = [a for a in c.get('asks') or () if a.get('kind') in ('board', 'explainer') and a.get('slug') in slugs and (a.get('at') or 0) >= now - UNCLEAR]
    for slug in dict.fromkeys(a['slug'] for a in sorted(asks, key=lambda a: a['at'])):
        mine = [a for a in asks if a['slug'] == slug]
        if len(mine) < 2: continue
        if any(a.get('correct') for a in lr.attempts(slug, since=max(a['at'] for a in mine))): continue
        return slugs.index(slug), len(mine)
    return None

In [ ]:
#| export
def _weak_later(lr, rows, k, by, now):
    "Rule 6: `(concept, row)` for a weak concept that you saw and that stage row `k` or a later one needs."
    names = {fold(x.name): x for x in by.values()}
    for r in rows[k:]:
        if r.state not in LEFT: continue
        need = [names[fold(i)] for i in r.get('ideas') or () if fold(i) in names] + ([by[s['slug']] for s in r.j.steps if s['slug'] in by] if r.j else [])
        need += [by[p] for x in list(need) for p in x.get('prereqs') or () if p in by]
        if (w := first(x for x in need if x.get('seen') and understanding(x, now).weak)): return w, r
    return None

def next_up(lr:Learning, c, rows=None, by:dict=None, now:float=None) -> AttrDict:
    "The one thing to do next in course `c`: `rule`, `why` (one line), `slug`, `name`, `stage` and `go`."
    now, by = now or time.time(), by if by is not None else {x.slug: x for x in lr.concepts()}
    rows = rows if rows is not None else stage_rows(lr, c, by)
    k = next((k for k, r in enumerate(rows) if r.state in LEFT), None)
    def out(rule, why, go, slug='', r=None): return AttrDict(rule=rule, why=why, go=go, slug=slug, name=_name(by, slug) if slug else '', stage=r['key'] if r else '')
    if k is None: return out(8, 'You reached your goal.', dict(kind='done'))
    r = rows[k]
    if r.ws:
        if (i := _leela_failed(lr, r)) is not None: return out(1, 'Your notebook did not pass. Look at the error first.', _step(r, i), r.j.steps[i]['slug'], r)
        if (x := _review(lr, r, by, now)) is not None: return out(2, f'Review {x.name} first. The next idea needs it, and you start to forget it.', dict(kind='review', slug=x.slug), x.slug, r)
        if (u := _unclear(lr, c, r, now)): return out(3, f"You asked about {_name(by, r.j.steps[u[0]]['slug'])} {_times(u[1])} times. Check it now.", _step(r, u[0]), r.j.steps[u[0]]['slug'], r)
        i = _here(c, r)
        chose = i is not None and i != r.ws.current
        i = i if i is not None else r.ws.current
        slug = r.j.steps[i]['slug']
        cc = by.get(slug)
        if cc is None or ready(cc, by): return out(4, 'You opened this idea last, so you go on from it.' if chose else f'Next: {_name(by, slug)}. You know the ideas that it needs.', _step(r, i), slug, r)
        p = min((by[p] for p in cc.get('prereqs') or () if p in by and by[p].get('floor', 0) < 3), key=lambda x: x.get('floor', 0))
        return out(5, f'You opened {cc.name} last. It needs {p.name} first.' if chose else f'Next: {p.name} first. {cc.name} needs it.', _step(r, i), slug, r)
    if (w := _weak_later(lr, rows, k, by, now)): return out(6, f"Practice {w[0].name}. Stage {w[1].n} builds on it.", dict(kind='review', slug=w[0].slug), w[0].slug, w[1])
    return out(7, f"Stage {r.n} is next: {r['name']}." + (f" It makes {r['makes']}." if r.get('makes') else ''), dict(kind='stage', key=r['key']), '', r)

### Each rule, with a small course

`fixture` makes a course with one goal walk of three ideas: Noise, then Denoise (it needs Noise), then Sample (it needs Denoise).
`done(i)` marks idea `i` done, with its concept at depth 4.

In [ ]:
def fixture():
    l = temp_learning()
    for n, pre in (('Noise', []), ('Denoise', ['noise']), ('Sample', ['denoise'])): l.put(n, f'{n} a picture.', 'ml', prereqs=pre)
    goal_walk(l); c = join_course(l, 'gw')
    return l, c
def done(l, i, at=None):
    slug = l.journey('gw').steps[i]['slug']
    l.walk_step('gw', i, done=dict(outcome='solid', at=at or time.time())); l.update(slug, floor=4, seen=1, last=time.time(), stability=5., due=time.time() + 5 * 86400)

**Rule 4.** Two ideas are done, so you are at idea 3, and you know what it needs.

In [ ]:
l, c = fixture(); done(l, 0); done(l, 1)
nx = next_up(l, l.course(c.id))
test_eq((nx.rule, nx.slug, nx.go), (4, 'sample', dict(kind='step', walk='gw', step=2)))
test_eq(nx.why, 'Next: Sample. You know the ideas that it needs.')

**Rule 4, you chose it.** You opened idea 3 while idea 2 is still to learn. Your pick wins, with its reason.

In [ ]:
l, c = fixture(); done(l, 0); l.update('denoise', floor=3)   # Denoise is ready to build on, but not learned
l.merge_course(c.id, here=dict(walk='gw', step=2, slug='sample', at=time.time()))
test_eq((next_up(l, l.course(c.id)).slug, next_up(l, l.course(c.id)).why), ('sample', 'You opened this idea last, so you go on from it.'))

**Rule 5.** Nothing is done, and idea 2 is next when you skip idea 1: Denoise needs Noise, which you do not know.

In [ ]:
l, c = fixture(); l.walk_step('gw', 0, done=dict(outcome='skipped', at=time.time()))
nx = next_up(l, l.course(c.id))
test_eq((nx.rule, nx.slug, nx.why), (5, 'denoise', 'Next: Noise first. Denoise needs it.'))

**Rule 1.** Leela filed a failed check for idea 3. That comes first.

In [ ]:
l, c = fixture(); done(l, 0); done(l, 1)
l.record('sample', 'Run the tests', correct=False, kind='code', app='leela', walk='gw', step=2)
nx = next_up(l, l.course(c.id))
test_eq((nx.rule, nx.go['step'], nx.why), (1, 2, 'Your notebook did not pass. Look at the error first.'))

**Rule 2.** Denoise is done, Sample needs it, and its review is due with a low recall. After one review, the course moves on.

In [ ]:
l, c = fixture(); done(l, 0, at=time.time() - 20 * 86400); done(l, 1, at=time.time() - 20 * 86400)
l.update('denoise', last=time.time() - 30 * 86400, stability=1., due=time.time() - 86400)
nx = next_up(l, l.course(c.id))
test_eq((nx.rule, nx.go, nx.why), (2, dict(kind='review', slug='denoise'), 'Review Denoise first. The next idea needs it, and you start to forget it.'))
l.record('denoise', 'What does denoise do?', correct=True)
test_eq(next_up(l, l.course(c.id)).rule, 4)

`add_ask` keeps one ask on the course: a board of the screen guide, an explainer, a demo or a notebook. It keeps only the words
that you said, the idea and the time, never the page.

In [ ]:
#| export
ASK_KINDS = ('board', 'explainer', 'demo', 'notebook', 'lesson')

def add_ask(lr:Learning, cid:str, where:str, slug:str, kind:str, said:str='', ref:str='', outcome:str='', at:float=None) -> AttrDict:
    "Keep one ask on course `cid`: `where` (walk, session or guide), the idea `slug`, its `kind`, your words `said`. At most `ASKS_MOST`, the newest."
    if kind not in ASK_KINDS: raise ValueError(f'{kind!r} is not a kind of ask')
    a = dict(at=at or time.time(), where=where, said=clip(' '.join(str(said or '').split()), 300), slug=slug, kind=kind, ref=ref, outcome=outcome)
    def f(c): c['asks'] = [*(c.get('asks') or ()), a][-ASKS_MOST:]
    return lr.change_course(cid, f)

**Rule 3.** You asked the screen guide about idea 3 two times. A right check after the last ask clears it.

In [ ]:
l, c = fixture(); done(l, 0); done(l, 1)
for _ in range(2): add_ask(l, c.id, 'guide', 'sample', 'board', said='what is sampling?', at=time.time() - 60)
nx = next_up(l, l.course(c.id))
test_eq((nx.rule, nx.why), (3, 'You asked about Sample two times. Check it now.'))
l.record('sample', 'Sample one picture', correct=True)
test_eq(next_up(l, l.course(c.id)).rule, 4)
test_fail(lambda: add_ask(l, c.id, 'guide', 'sample', 'page'), contains='kind of ask')

**Rules 6, 7 and 8.** With the goal walk done and a planned stage after it, the next stage comes, after a weak idea that it needs. With no stage left, you reached your goal.

In [ ]:
l, c = fixture(); [done(l, i) for i in range(3)]
l.change_course(c.id, lambda c: c.stages.append(dict(key='s2', name='Your comic data', makes='captions.jsonl', ideas=['Sample', 'Captions'], walk=None, status='planned')))
test_eq(next_up(l, l.course(c.id)).why, 'Stage 2 is next: Your comic data. It makes captions.jsonl.')
l.update('sample', misses=2)
nx = next_up(l, l.course(c.id)); test_eq((nx.rule, nx.why), (6, 'Practice Sample. Stage 2 builds on it.'))
l.change_course(c.id, lambda c: c.stages.pop())
test_eq(next_up(l, l.course(c.id)).why, 'You reached your goal.')

### What stands between you and the goal

`blocking` gives the rows, in this order, each with a line and one action:

- **idea**: the ideas still to learn in the stages that are not finished;
- **weak**: an idea that you learned, that is weak now, and that a stage still to do needs;
- **notebook**: notebook ideas with no passed check from Leela;
- **setup**: Leela is not running (when the block knows it, `leela=False`), or a notebook has no folder for its data;
- **source**: a source that failed or still files, ideas that research still looks for, and a stage with no passage for half of its ideas.

In [ ]:
#| export
def _names(ns, most=3) -> str: return ', '.join(ns[:most]) + (f', and {len(ns) - most} more' if len(ns) > most else '')

def blocking(lr:Learning, c, rows=None, by:dict=None, leela:bool=None, now:float=None) -> list:
    "The rows of \"what stands between you and the goal\": `{kind, line, act, go}`, in the order of the kinds."
    now, by = now or time.time(), by if by is not None else {x.slug: x for x in lr.concepts()}
    rows, out = rows if rows is not None else stage_rows(lr, c, by), []
    live = [r for r in rows if r.state in LEFT]
    ideas = [_name(by, r.j.steps[i]['slug']) for r in live for i in _left(r)] + [i for r in live if not r.j for i in r.get('ideas') or ()]
    nb = [(r, i) for r in live for i in _left(r) if r.j.steps[i]['check'].get('kind') == 'notebook']
    if ideas: out.append(AttrDict(kind='idea', line=f"{_n(len(ideas), 'idea')}: {_names(ideas)}.", act='Next idea', go=None))
    if live and (w := _weak_later(lr, rows, rows.index(live[0]), by, now)):
        out.append(AttrDict(kind='weak', line=f"{w[0].name} is not solid yet: {(understanding(w[0], now).reasons or ['you missed it'])[0]}.", act='Practice it', go=dict(kind='review', slug=w[0].slug)))
    if nb: out.append(AttrDict(kind='notebook', line=f"{_n(len(nb), 'notebook')} to run in Leela: {_names([_name(by, r.j.steps[i]['slug']) for r, i in nb])}.", act='Open it', go=_step(*nb[0])))
    if nb and leela is False: out.append(AttrDict(kind='setup', line='Leela is not running. Start Leela to run the notebooks.', act='', go=None))
    for r, i in nb:
        for d in (r.j.steps[i].get('nb') or {}).get('data') or ():
            if not str(d.get('path') or '').strip(): out.append(AttrDict(kind='setup', line=f"Set the folder of {d.get('what') or d.get('name')}.", act='Set it', go=_step(r, i)))
    for s in c.get('sources') or ():
        if s.get('status') == 'failed': out.append(AttrDict(kind='source', line=s.get('why') or f"abhyasa could not file {s['root']}.", act='', go=None))
        elif s.get('status') == 'filing': out.append(AttrDict(kind='source', line=f"abhyasa still files {s['root']}.", act='', go=None))
    for r in live:
        if not r.j: continue
        steps = [r.j.steps[i] for i in _left(r)]
        if (n := sum(s.get('research') == 'pending' for s in steps)): out.append(AttrDict(kind='source', line=f"Stage {r.n}: abhyasa looks for a source for {_n(n, 'idea')}.", act='', go=None))
        elif steps and (n := sum(not (s.get('read') or {}).get('text') for s in steps)) * 2 >= len(steps):
            out.append(AttrDict(kind='source', line=f"Stage {r.n}: {_n(n, 'idea')} {'has' if n == 1 else 'have'} no source yet.", act='', go=_step(r, _left(r)[0])))
    return out

A failed clone shows a `source` row. A notebook idea with Leela off shows a `notebook` row and a `setup` row, and its empty data folder a second `setup` row.

In [ ]:
l, c = fixture()
l.walk_slug('gw', 'sample', check=dict(way='code', kind='notebook', depth=4), nb=dict(data=[dict(name='comics', what='your comic images', path='')]))
bs = blocking(l, l.course(c.id), leela=False)
test_eq([b.kind for b in bs], ['idea', 'notebook', 'setup', 'setup', 'source', 'source'])
test_eq(bs[0].line, '3 ideas: Noise, Denoise, Sample.')
test_eq([b.line for b in bs[1:5]], ['1 notebook to run in Leela: Sample.', 'Leela is not running. Start Leela to run the notebooks.',
                                    'Set the folder of your comic images.', 'abhyasa could not clone huggingface/diffusers.'])
test_eq(bs[5].line, 'Stage 1: 3 ideas have no source yet.')

### `course_state`

`course_state` puts it together for the page, with no write: the course, the stages, the current stage, "you are here", the
counts over all stages (a planned stage counts by its plan), next and why, the blocking rows, what you made in Leela, and
`finished`.

In [ ]:
#| export
def course_state(lr:Learning, cid:str, leela:bool=None, now:float=None) -> AttrDict|None:
    "Where you are in course `cid`, what comes next and why, and what stands between you and the goal. It writes nothing. None for an unknown course."
    if (c := lr.course(cid)) is None: return None
    now, by = now or time.time(), {x.slug: x for x in lr.concepts()}
    rows = stage_rows(lr, c, by)
    nx = next_up(lr, c, rows, by, now)
    cur = first(r for r in rows if r.state in LEFT)
    here = None
    if cur and cur.ws:
        i = nx.go['step'] if nx.go.get('kind') == 'step' and nx.go.get('walk') == cur.j.id else _here(c, cur) if _here(c, cur) is not None else cur.ws.current
        if i is not None: here = AttrDict(walk=cur.j.id, step=i, slug=cur.j.steps[i]['slug'], name=_name(by, cur.j.steps[i]['slug']), n=i + 1, of=len(cur.j.steps), stage=cur.n)
    made = [AttrDict(stage=r.n, slug=r.j.steps[i]['slug'], name=_name(by, r.j.steps[i]['slug']), images=len(b.run.get('images') or ()), at=b.run.get('at'))
            for r in rows if r.j for i, b in sorted(leela_found(lr, r.j).items()) if b.run and b.run.get('images')]
    counted = [r for r in rows if r.state != 'skipped']
    return AttrDict(course=c, stages=rows, current=cur, here=here, next=nx, done=sum(r.done for r in counted), total=sum(r.total for r in counted),
                    blocking=blocking(lr, c, rows, by, leela, now), made=made, finished=nx.rule == 8 and bool(rows), level=level_done(c))

The state of the fixture after two ideas: you are at idea 3 of 3 in stage 1, and two of three ideas are done.

In [ ]:
l, c = fixture(); done(l, 0); done(l, 1)
st = course_state(l, c.id)
test_eq((st.here.n, st.here.of, st.here.stage, st.here.name), (3, 3, 1, 'Sample'))
test_eq((st.done, st.total, st.next.rule, st.finished), (2, 3, 4, False))
done(l, 2); test_eq(course_state(l, c.id).finished, True)
test_is(course_state(l, 'none'), None)

## Known ideas

An idea that you learned in a session or in another walk counts: it shows "known (learned elsewhere)", it asks nothing, and its
review comes when it is due. Here you learn Denoise in a session. The course counts it, and "you are here" goes past it.

In [ ]:
l, c = fixture(); done(l, 0)
l.update('denoise', floor=4, seen=1)   # a session closed Denoise at depth 4
st = course_state(l, c.id)
test_eq([s.dot for s in st.stages[0].ws.steps], ['done', 'known', 'now'])
test_eq((st.stages[0].ws.steps[1].elsewhere, st.done, st.here.name), (True, 2, 'Sample'))

A new goal walk marks an idea that you learned as `known` when it is made, so it never asks it.

In [ ]:
l = temp_learning(); l.put('Noise', 'n', 'ml'); l.update('noise', floor=4, seen=1)
wj = walk_from_topic(l, Fake({'You plan a learning path': dict(title='T', subject='ml', steps=[dict(name='Noise', statement='n', why='w', slug='noise'), dict(name='Captions', statement='c', why='w')])}), 'Make comics')
test_eq([s['status'] for s in wj.steps], ['known', 'todo'])

## Where you are

`set_here` keeps the last idea that you opened in a walk of the course. It writes only when the idea changed, or after a minute,
so a page that loads again does not write each time. `active_course` is the course that you used last, for Today.

In [ ]:
#| export
def set_here(lr:Learning, j, i:int, now:float=None) -> AttrDict|None:
    "Keep step `i` of walk `j` as the place where you are in its course. None when the walk is in no course."
    now = now or time.time()
    if not (j.get('course') and 0 <= i < len(j['steps'])) or (c := course_of(lr, j['id'])) is None: return None
    h, slug = c.get('here') or {}, j['steps'][i]['slug']
    if (h.get('walk'), h.get('slug')) == (j['id'], slug) and now - (h.get('at') or 0) < 60: return c
    return lr.merge_course(c.id, here=dict(walk=j['id'], step=i, slug=slug, at=now))

def active_course(lr:Learning) -> AttrDict|None:
    "The course that you used last: the newest `here`, else the newest course."
    return first(sorted(lr.courses(), key=lambda c: -max((c.get('here') or {}).get('at') or 0, c.get('created') or 0)))

In [ ]:
l, c = fixture()
test_eq(set_here(l, l.journey('gw'), 1).here['slug'], 'denoise')
v = l.course(c.id).v; set_here(l, l.journey('gw'), 1); test_eq(l.course(c.id).v, v)   # the same idea: no write
test_eq(active_course(l).id, c.id)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()